# 02 — Evaluate the trained model

**Purpose:** load the checkpoint from `01_train.ipynb`, compute held-out test metrics on chromosomes 1 and 2, and produce the numbers and plots for the README.

This notebook is a **thin consumer** of `src/evaluate.py`, all metric logic lives in `src/`. The notebook's job is to make the results legible.

**Metric definitions (for the README):**
- **AUROC** — area under the ROC curve. Threshold-independent ranking quality. 0.5 = chance, 1.0 = perfect.
- **AUPRC** — area under the precision-recall curve. With balanced classes (1:1), it should track AUROC closely.
- **Accuracy / precision / recall** — at a fixed decision threshold (default 0.5).

**Held-out set:** chromosomes `1` and `2`, never seen during training.

**Expected results:** a 1D CNN on CREB1 ChIP-seq typically achieves AUROC **0.85–0.95** on held-out chromosomes. AUROC near 0.5 means the model didn't learn, go back to `00_data_sanity.ipynb` before touching hyperparameters. AUROC > 0.98 is suspicious, check for train/test leakage.

**Note on the training run this evaluates:** the 20-epoch run showed val loss minimum at epoch 7, with overfitting after. The checkpoint saved is the **epoch-7 best-val weights**, not the epoch-20 weights. This is intentional and correct.

In [ ]:
import sys, os
from pathlib import Path

# Make `src` importable regardless of Jupyter's CWD
p = Path.cwd()
while p != p.parent and not (p / "src").exists():
    p = p.parent
os.chdir(p)
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
print("repo root:", p)

%load_ext autoreload
%autoreload 2

import json
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset

from src.data import DataConfig, build_dataset, split_by_chromosome
from src.model import Creb1CNN
from src.train import load_checkpoint
from src.evaluate import predict, compute_metrics, curves, evaluate_by_chromosome

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

## 1. Configuration and checkpoint

Same `DataConfig` and `test_chroms` as `01_train.ipynb`, **consistency here is critical.** If you change `test_chroms` between training and evaluation, you're evaluating on data the model trained on.

`load_checkpoint` returns a dict with `state_dict` plus metadata (`epoch`, `val_loss`) saved during training. The `epoch` tells you which training epoch produced the best-val weights.

In [ ]:
cfg = DataConfig(window=500, neg_per_pos=1, seed=0, test_chroms=("1", "2"))
CKPT = "checkpoints/cnn.pt"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
THRESHOLD = 0.5

model = Creb1CNN()
meta = load_checkpoint(CKPT, model)
print(f"checkpoint: {CKPT}")
print(f"  best epoch:    {meta.get('epoch')}")
print(f"  best val loss: {meta.get('val_loss'):.4f}")
print(f"  device:        {DEVICE}")

## 2. Load the held-out test set

`split_by_chromosome` with `test_chroms=("1", "2")` produces a test set from **only those chromosomes**. Every row in `X_te` comes from a chromosome the model never saw during training — this is the leak-free evaluation that makes the AUROC meaningful.

**What to check:** `positives` and `negatives` counts should be roughly balanced (the test set carries the 1:1 pos/neg ratio from training data).

In [ ]:
data = build_dataset(cfg)
split = split_by_chromosome(data, test_chroms=cfg.test_chroms)

X_te = split["X_test"]
y_te = split["y_test"]
chroms_te = split["chroms_test"]

print(f"test: {X_te.shape}")
print(f"positives: {int(y_te.sum()):,}   negatives: {int((1-y_te).sum()):,}")
print(f"chromosomes: {sorted(set(chroms_te))}")

## 3. Predict on the test set

`predict()` runs the model in `eval()` mode (dropout off), applies sigmoid to the logits, and returns `(y_true, y_prob)`. The sigmoid is applied **only here** — the model emits raw logits by design, and `compute_metrics` expects probabilities.

**What to check:** the printed `y_prob` range should be `[0, 1]`. If you see values outside that range, sigmoid was applied twice (in model and in predict) or not at all.

In [ ]:
loader = DataLoader(
    TensorDataset(torch.from_numpy(X_te).float(), torch.from_numpy(y_te).float()),
    batch_size=512, shuffle=False,
)

y_true, y_prob = predict(model, loader, device=DEVICE)

print(f"y_true shape: {y_true.shape}")
print(f"y_prob shape: {y_prob.shape}")
print(f"y_prob range: [{y_prob.min():.4f}, {y_prob.max():.4f}]")
print(f"y_prob mean:  {y_prob.mean():.4f}")
print(f"y_true mean:  {y_true.mean():.4f}")

## 4. Overall metrics on held-out chromosomes

This is the **headline result**. Copy these numbers into `readme.md`.

**How to read them:**
- `auroc` — the primary metric. 0.85+ is good; 0.90+ is strong.
- `auprc` — should be close to AUROC for balanced classes. A large gap indicates poor calibration at extremes.
- `accuracy` — threshold-dependent. At AUROC 0.90, expect accuracy 0.80–0.85 at threshold 0.5.
- `precision` / `recall` — at threshold 0.5, roughly equal when classes are balanced.

**If AUROC < 0.75:** stop, don't tune hyperparameters. Run `00_data_sanity.ipynb` — the bug is almost certainly upstream (data pipeline or train/test split).

In [ ]:
metrics = compute_metrics(y_true, y_prob, THRESHOLD)

print("=" * 40)
print("OVERALL (held-out chr1 + chr2)")
print("=" * 40)
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k:12s} {v:.4f}")
    else:
        print(f"  {k:12s} {v}")

## 5. ROC and precision-recall curves

The plots to include in the README.

- **ROC** — true positive rate vs false positive rate across all thresholds. The diagonal is chance. A good model bows toward the top-left.
- **PR** — precision vs recall. The horizontal dashed line is the positive rate (chance level for a random classifier).

**What to check:** the curves should stay well away from the chance lines across the full threshold range, not just at one operating point.

In [ ]:
c = curves(y_true, y_prob)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ROC
axes[0].plot(c["fpr"], c["tpr"], lw=2, color="C0")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4, label="chance")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].set_title(f"ROC  (AUROC = {metrics['auroc']:.3f})")
axes[0].legend(loc="lower right")
axes[0].grid(alpha=0.3)

# PR
axes[1].plot(c["recall"], c["precision"], lw=2, color="C1")
axes[1].axhline(y=float(y_true.mean()), color="k", linestyle="--", alpha=0.4,
                label=f"chance ({y_true.mean():.2f})")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title(f"PR  (AUPRC = {metrics['auprc']:.3f})")
axes[1].legend(loc="lower left")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Per-chromosome breakdown

Aggregate AUROC can hide a chromosome-specific failure — e.g. the model works on chr1 but fails on chr2, which would suggest a chromosome-specific artifact (different peak density, GC content, or repeat structure).

**What to check:**
- AUROC should be roughly consistent across chr1 and chr2 (within ~0.05).
- A large gap (e.g. 0.90 on chr1, 0.65 on chr2) suggests the model learned chromosome-specific features rather than the actual binding motif.
- If either chromosome shows "single-class", it means after filtering there was only one class of test rows on that chromosome. Not a bug — just too few peaks there.

In [ ]:
per_chrom = evaluate_by_chromosome(y_true, y_prob, chroms_te, THRESHOLD)

rows = []
for chrom, m in per_chrom.items():
    if "auroc" in m:
        rows.append({
            "chrom": chrom,
            "n": m["n"],
            "auroc": round(m["auroc"], 4),
            "auprc": round(m["auprc"], 4),
            "accuracy": round(m["accuracy"], 4),
        })
    else:
        rows.append({
            "chrom": chrom,
            "n": m["n"],
            "auroc": None,
            "auprc": None,
            "accuracy": None,
        })

df = pd.DataFrame(rows).sort_values("chrom").reset_index(drop=True)
display(df)

## 7. Confusion matrix at the decision threshold

Makes the precision/recall tradeoff concrete at the threshold you'd use to actually call peaks.

- **Top-left**: true negatives (correctly not-peak)
- **Bottom-right**: true positives (correctly peak)
- **Top-right**: false positives (overcalled) — CREB1 ChIP-seq peaks you'd call that aren't real
- **Bottom-left**: false negatives (missed peaks) — real CREB1 peaks you'd fail to call

**What to check:** neither off-diagonal should dominate. A model with 90% accuracy but 50% false negatives is useless for peak calling, even though the accuracy looks great.

In [ ]:
try:
    import seaborn as sns
    HAS_SNS = True
except ImportError:
    HAS_SNS = False

from sklearn.metrics import confusion_matrix

y_pred = (y_prob >= THRESHOLD).astype(int)
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

fig, ax = plt.subplots(figsize=(4.5, 3.8))
if HAS_SNS:
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["pred neg", "pred pos"],
                yticklabels=["true neg", "true pos"], ax=ax)
else:
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center")
    ax.set_xticks([0, 1], ["pred neg", "pred pos"])
    ax.set_yticks([0, 1], ["true neg", "true pos"])
ax.set_title(f"Confusion matrix @ threshold={THRESHOLD}")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"TP: {tp:,}   FP: {fp:,}   TN: {tn:,}   FN: {fn:,}")
print(f"False positive rate: {fp / (fp + tn):.4f}")
print(f"False negative rate: {fn / (fn + tp):.4f}")

## 8. Threshold sweep 

The default threshold of 0.5 is rarely optimal for a real application. Depending on whether you care more about false positives (strict threshold) or false negatives (loose threshold), the right operating point may be anywhere in [0.1, 0.9].

This table shows accuracy / precision / recall at several thresholds so you can pick one for the README based on the intended use case.

**For a peak-caller context:** you usually want **high recall** (don't miss real peaks), accepting more false positives. A threshold of 0.3–0.4 is typical.

In [ ]:
rows = []
for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    m = compute_metrics(y_true, y_prob, threshold=t)
    rows.append({
        "threshold": t,
        "accuracy": round(m["accuracy"], 4),
        "precision": round(m["precision"], 4),
        "recall": round(m["recall"], 4),
    })

display(pd.DataFrame(rows))

## 9. Save results to JSON

`data/processed/metrics.json` is the artifact the README cites and the file you diff against when you change something. It's tiny (~2 KB), deterministic given a checkpoint, and contains everything needed to reproduce the headline numbers.

**Do commit this file.** It's a record of the result, not data.

In [ ]:
out = Path("data/processed/metrics.json")
out.parent.mkdir(parents=True, exist_ok=True)

out.write_text(json.dumps({
    "overall": metrics,
    "per_chromosome": per_chrom,
    "checkpoint": {
        "path": str(CKPT),
        "epoch": meta.get("epoch"),
        "val_loss": meta.get("val_loss"),
    },
    "config": {k: str(v) if isinstance(v, Path) else v
               for k, v in cfg.__dict__.items()},
}, indent=2, default=str))

print(f"wrote {out}")
print(f"size: {out.stat().st_size:,} bytes")

## 10. Inspect low- and high-confidence predictions

Eyeballing a few sequences catches label noise (a "negative" that visibly contains a CREB1 motif) and distribution shift (a "positive" with no recognizable motif).

The sequences below are the 4 lowest-confidence and 4 highest-confidence predictions. **What to look for:**
- High-confidence positives should contain something resembling the CREB1 motif (`TGACGTCA` or close variants like `TGACGTAA`, `CGACGTCA`).
- If high-confidence positives look random, the model is exploiting an artifact (GC content, repeat structure) rather than the actual binding motif — a warning sign even if AUROC looks good.
- Low-confidence predictions are near the decision boundary — they're the ones a threshold change would flip.

In [ ]:
order = np.argsort(y_prob)
n_show = 4

fig, axes = plt.subplots(2, n_show, figsize=(3 * n_show, 4))

for col, idx in enumerate(np.concatenate([order[:n_show], order[-n_show:]])):
    r, c_ = divmod(col, n_show)
    x = X_te[idx]
    seq = "".join("ACGT"[int(np.argmax(x[:, i]))] if x[:, i].any() else "N"
                  for i in range(x.shape[1]))
    axes[r, c_].text(0, 0.5, seq[:50] + "...", fontsize=6, family="monospace")
    axes[r, c_].set_title(f"true={int(y_true[idx])}  prob={y_prob[idx]:.3f}",
                          fontsize=9)
    axes[r, c_].axis("off")

plt.suptitle("Lowest (top) and highest (bottom) confidence predictions")
plt.tight_layout()
plt.show()

In [ ]:
def to_seq(x):
    return "".join("ACGT"[int(np.argmax(x[:, i]))] if x[:, i].any() else "N"
                   for i in range(x.shape[1]))

motifs = ["TGACGTCA", "TGACGTAA", "CGACGTCA", "TGACGTCG"]
n_check = min(2000, len(X_te))

print(f"Motif enrichment over first {n_check} test samples:")
print(f"{'motif':12s} {'pos_hits':>10s} {'neg_hits':>10s} {'enrichment':>12s}")
print("-" * 48)

pos_seqs = [to_seq(X_te[i]) for i in range(n_check) if y_true[i] == 1]
neg_seqs = [to_seq(X_te[i]) for i in range(n_check) if y_true[i] == 0]

for motif in motifs:
    pos_hits = sum(motif in s for s in pos_seqs)
    neg_hits = sum(motif in s for s in neg_seqs)
    rate_pos = pos_hits / max(len(pos_seqs), 1)
    rate_neg = neg_hits / max(len(neg_seqs), 1)
    enrich = rate_pos / rate_neg if rate_neg > 0 else float("inf")
    print(f"{motif:12s} {pos_hits:>10d} {neg_hits:>10d} {enrich:>11.2f}x")

**Interpreting the motif enrichment table:**

The exact CREB1 consensus (`TGACGTCA`) is short (8 bp) and degenerate in practice — expect it to appear in maybe 5–20% of positive sequences and ~1–5% of negatives. The **enrichment ratio** (`pos_hits_rate / neg_hits_rate`) is the number that matters:

- **≥ 3×** — model's positives are enriched for the motif. Good sign the data is real.
- **1–2×** — weak enrichment. Could mean the motif is degenerate in ChIP-seq peaks (common) or the summit coordinate is off.
- **≈ 1×** — no enrichment. The "positives" in the test set don't actually contain CREB1 sites. That's a data problem worth investigating before trusting the AUROC.

This is a *data sanity* check, not a *model* check. It tells you whether the test set is what you think it is. A model can have AUROC 0.90 while the motifs are barely enriched — that would mean the model learned something else (GC content, repeat structure), which is a known failure mode for DNA-binding CNNs.

If enrichment is weak, the previous `00_data_sanity.ipynb` check (running the same motif count on `X_pos` vs `X_neg`) should also have shown it. If that check passed and this one doesn't, something is wrong with `split_by_chromosome` — but that's unlikely, since `_peak_center` and `extract_window` don't change between notebooks.

## Done

Everything found back in the `readme.md`:
- Headline AUROC / AUPRC numbers (cell 10)
- ROC and PR curves (cell 12)
- Per-chromosome breakdown (cell 14)
- Confusion matrix at threshold 0.5 (cell 16)
- Threshold sweep for real-world operating points (cell 18)
- `data/processed/metrics.json` as a durable record (cell 20)

**Final checklist before shipping:**
- [ ] AUROC ≥ 0.85 (if not, investigate data, don't tune hyperparameters)
- [ ] AUROC < 0.98 (if higher, check for train/test leakage — chr1 or chr2 accidentally in training)
- [ ] Per-chromosome AUROC consistent within ~0.05 (if not, chromosome-specific artifact)
- [ ] Motif enrichment ≥ 2× (if not, model may have learned GC content rather than CREB1 binding)
- [ ] `metrics.json` committed to git

**If AUROC is unexpectedly high (> 0.98):** be suspicious, not excited. Almost-certainly-too-good usually means train/test leakage. Re-check `split_by_chromosome` on the test set — every row in `X_te` must come from chr1 or chr2, and no row from any other chromosome.

**If AUROC is unexpectedly low (< 0.75):** the problem is data, not the model. Go back to `00_data_sanity.ipynb` and re-run every check — particularly the chromosome overlap and motif enrichment checks.